# Internal AMP Classifier + Condition Adherence for cVAE

This notebook trains an internal multi-label AMP classifier and uses it to evaluate how well generated peptides follow requested activity conditions.

Main goals:
1. load the same preprocessed peptide dataset used in the project,
2. train a sequence-based multi-label classifier,
3. tune decision thresholds under label imbalance,
4. reload the trained cVAE generator,
5. score generated peptides for condition adherence,
6. rank promising candidates for downstream screening.

This classifier is internal: it is trained on our project dataset and serves as a practical evaluation tool, not as ground truth.


## Notebook logic

This notebook has two parts.

First, we train and validate a multi-label peptide classifier on the labeled AMP dataset.
Second, we use that classifier as an internal oracle to score peptides generated by the cVAE under different requested conditions.

This separation is useful because generation quality and condition adherence should be evaluated independently from the generator loss alone.

In [ ]:
# Core imports
import os
import math
import json
import random
import pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef,
    balanced_accuracy_score,
)

SEED = 42

# Fix random seeds so data split, training, and downstream adherence
# estimates remain as reproducible as possible across runs.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

# Prefer GPU or Apple MPS when available, but keep the notebook runnable on CPU.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("device:", device)

In [ ]:
# Keep all classifier artifacts under model_training/models so they can be reused
# by later notebooks without retraining.
ROOT = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path(os.getcwd()).parent
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

DATA_PATH = DATA_DIR / "preprocessed" / "data.csv"
VOCAB_PATH = MODELS_DIR / "vocab.pkl"

print("ROOT:", ROOT)
print("DATA_PATH:", DATA_PATH)
print("MODELS_DIR:", MODELS_DIR)
print("VOCAB_PATH:", VOCAB_PATH)

## 1. Load data
The notebook expects the same preprocessed APD-style CSV as the generator notebook.

This section checks that the label columns are present and gives a quick view of sequence length statistics before any modeling starts.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"{DATA_PATH} not found. Run preprocessing first.")

df = pd.read_csv(DATA_PATH)

seq_col = "Sequence"
len_col = "Length"

# The first five labels are the main study targets.
# Antiviral and antiparasitic are kept as bonus labels for broader multi-label testing.
target_cols = [
    "is_antibacterial",
    "is_anti_gram_positive",
    "is_anti_gram_negative",
    "is_antifungal",
    "is_anticancer",
    "is_antiviral",       # bonus
    "is_antiparasitic",   # bonus
]

required_cols = [seq_col] + target_cols
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in dataset: {missing}")

df = df[required_cols + ([len_col] if len_col in df.columns else [])].dropna(subset=[seq_col]).copy()
# Normalize sequences to uppercase strings so tokenization is deterministic.
df[seq_col] = df[seq_col].astype(str).str.strip().str.upper()

for c in target_cols:
    df[c] = df[c].fillna(0).astype(int)

if len_col not in df.columns:
    df[len_col] = df[seq_col].str.len()

# Remove empty sequences because they are not meaningful classifier inputs.
df = df[(df[seq_col].str.len() > 0)].reset_index(drop=True)

print("Dataset size:", df.shape)
display(df.head())

print("\nLabel prevalence:")
for c in target_cols:
    print(f"{c:24s} {df[c].sum():5d} / {len(df):5d} = {df[c].mean():.3f}")

In [ ]:
# Basic sequence length summary
display(df[len_col].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("length"))

## 2. Build or reload the tokenizer

To make classifier inputs compatible with the generator pipeline, we reuse the saved vocabulary when available.
Otherwise, we rebuild a character-level tokenizer directly from the current dataset.

This keeps the classifier self-contained while preserving alignment with the rest of the project.

In [ ]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"

if VOCAB_PATH.exists():
    with open(VOCAB_PATH, "rb") as f:
        vocab_obj = pickle.load(f)
    char2idx = vocab_obj["char2idx"]
    idx2char = vocab_obj["idx2char"]
    max_len = int(vocab_obj["max_len"])
    print("Loaded tokenizer from vocab.pkl")
else:
    alphabet = sorted({aa for seq in df[seq_col] for aa in seq})
    vocab_list = [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN] + alphabet
    char2idx = {ch: i for i, ch in enumerate(vocab_list)}
    idx2char = {i: ch for ch, i in char2idx.items()}
    max_len = int(min(max(df[len_col].max() + 2, 16), 64))
    print("Built tokenizer from data")

PAD_IDX = char2idx[PAD_TOKEN]
UNK_IDX = char2idx[UNK_TOKEN]
SOS_IDX = char2idx[SOS_TOKEN]
EOS_IDX = char2idx[EOS_TOKEN]
vocab_size = len(char2idx)

print("vocab_size:", vocab_size)
print("max_len:", max_len)

In [ ]:
def tokenize(seq, max_len=max_len):
    """
    Convert a peptide string into a fixed-length token sequence.

    We add explicit start and end tokens so the classifier input format stays
    compatible with the generator-side tokenization used elsewhere in the project.
    """
    tokens = [SOS_IDX] + [char2idx.get(ch, UNK_IDX) for ch in seq] + [EOS_IDX]
    seq_len = min(len(tokens), max_len)
    tokens = tokens[:max_len]
    if len(tokens) < max_len:
        tokens = tokens + [PAD_IDX] * (max_len - len(tokens))
    return tokens, seq_len

## 3. Train/validation/test split

We use a deterministic random split for a fast baseline.

This is acceptable for an internal screening model, but it is not a strong biological evaluation protocol because related peptides may still appear across splits.
A stronger future version should use a homology-aware split.


In [ ]:
idx = np.arange(len(df))
rng = np.random.default_rng(SEED)
rng.shuffle(idx)

n = len(idx)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

train_idx = idx[:n_train]
val_idx = idx[n_train:n_train + n_val]
test_idx = idx[n_train + n_val:]

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print("train:", train_df.shape, "val:", val_df.shape, "test:", test_df.shape)

In [ ]:
class MultiLabelAMPDataset(Dataset):
    """Dataset of peptide sequences and their multi-label activity targets."""
    def __init__(self, frame):
        self.sequences = frame[seq_col].tolist()
        self.labels = frame[target_cols].values.astype(np.float32)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        toks, seq_len = tokenize(seq)
                # Return both tokenized inputs and the raw sequence string so the same
        # batch can later be reused for qualitative inspection if needed.
        return (
            torch.tensor(toks, dtype=torch.long),
            torch.tensor(seq_len, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float32),
            seq,
        )

train_ds = MultiLabelAMPDataset(train_df)
val_ds = MultiLabelAMPDataset(val_df)
test_ds = MultiLabelAMPDataset(test_df)

batch_size = 128
# Shuffle only the training loader; validation and test must stay deterministic.
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

## 4. Internal classifier

Baseline architecture:
- token embedding
- BiLSTM encoder
- masked mean and max pooling
- MLP prediction head
- multi-label BCE loss with `pos_weight`

The goal is not to build the strongest classifier possible. The goal is to build a separate evaluator that does not share weights with the generator and can be used for fast internal screening.


In [ ]:
class InternalAMPClassifier(nn.Module):
    """
    Sequence classifier for multi-label AMP activity prediction.

    The model uses a bidirectional LSTM encoder followed by masked mean and max pooling.
    Mean pooling captures global sequence context, while max pooling highlights strong local signals.
    """
    def __init__(self, vocab_size, num_labels, embed_dim=128, hidden_dim=256, num_layers=1, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.dropout = nn.Dropout(dropout)
        self.encoder = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.0 if num_layers == 1 else dropout,
        )
        self.norm = nn.LayerNorm(hidden_dim * 4)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, num_labels),
        )

    def forward(self, x, lengths):
        emb = self.dropout(self.embedding(x))
        # Pack variable-length sequences so the LSTM ignores pure padding positions.
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu().clamp(min=1), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.encoder(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=x.size(1))

        # Mask out padded positions before pooling so they do not distort the representation.
        mask = (x != PAD_IDX).unsqueeze(-1)  # [B, T, 1]
        out_masked = out.masked_fill(~mask, 0.0)

        denom = mask.sum(dim=1).clamp(min=1)
        mean_pool = out_masked.sum(dim=1) / denom

        neg_inf = torch.finfo(out.dtype).min
        max_pool = out.masked_fill(~mask, neg_inf).max(dim=1).values
        max_pool = torch.where(torch.isfinite(max_pool), max_pool, torch.zeros_like(max_pool))

        # Combine global average information with the strongest local activations.
        feats = torch.cat([mean_pool, max_pool], dim=1)
        feats = self.norm(feats)
        logits = self.head(feats)
        return logits

In [ ]:
# Class imbalance weights
y_train = train_df[target_cols].values.astype(np.float32)
pos_counts = y_train.sum(axis=0)
neg_counts = len(y_train) - pos_counts
# Multi-label AMP data is imbalanced, so positive labels for rare classes
# receive larger weight in BCEWithLogitsLoss.
pos_weight = np.clip(neg_counts / np.clip(pos_counts, 1, None), 1.0, 50.0)

pos_weight_t = torch.tensor(pos_weight, dtype=torch.float32, device=device)
print(pd.DataFrame({"label": target_cols, "pos_weight": pos_weight}))

In [ ]:
model = InternalAMPClassifier(vocab_size=vocab_size, num_labels=len(target_cols)).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)

model

In [ ]:
def compute_multilabel_metrics(y_true, y_prob, threshold=0.5):
    """
    Compute per-label and aggregate metrics for a multi-label classifier.

    AUROC and AUPRC measure ranking quality, while F1/precision/recall/MCC
    measure thresholded decision quality. This distinction matters because
    downstream adherence uses binary label decisions, not only ranking scores.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    rows = []
    for i, label in enumerate(target_cols):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        ys = y_prob[:, i]

        if len(np.unique(yt)) < 2:
            auroc = np.nan
            auprc = np.nan
        else:
            auroc = roc_auc_score(yt, ys)
            auprc = average_precision_score(yt, ys)

        f1 = f1_score(yt, yp, zero_division=0)
        prec = precision_score(yt, yp, zero_division=0)
        rec = recall_score(yt, yp, zero_division=0)

        try:
            mcc = matthews_corrcoef(yt, yp)
        except Exception:
            mcc = np.nan

        try:
            bal_acc = balanced_accuracy_score(yt, yp)
        except Exception:
            bal_acc = np.nan

        rows.append({
            "label": label,
            "auroc": auroc,
            "auprc": auprc,
            "f1": f1,
            "precision": prec,
            "recall": rec,
            "mcc": mcc,
            "balanced_acc": bal_acc,
            "prevalence": yt.mean(),
        })

    metrics_df = pd.DataFrame(rows)

    macro = {
        "macro_auroc": np.nanmean(metrics_df["auroc"]),
        "macro_auprc": np.nanmean(metrics_df["auprc"]),
        "macro_f1": np.nanmean(metrics_df["f1"]),
        "macro_precision": np.nanmean(metrics_df["precision"]),
        "macro_recall": np.nanmean(metrics_df["recall"]),
        "macro_mcc": np.nanmean(metrics_df["mcc"]),
        "macro_balanced_acc": np.nanmean(metrics_df["balanced_acc"]),
    }

    # micro F1 on flattened predictions
    macro["micro_f1"] = f1_score(y_true.reshape(-1), y_pred.reshape(-1), zero_division=0)

    return metrics_df, macro


@torch.no_grad()
def predict_loader(model, loader):
    """
    Run inference on a data loader and return loss, true labels,
    predicted probabilities, and raw peptide sequences.
    """
    model.eval()
    probs_all, labels_all, seqs_all = [], [], []
    total_loss, total_n = 0.0, 0
    for x, lengths, y, seqs in loader:
        x = x.to(device)
        lengths = lengths.to(device)
        y = y.to(device)

        logits = model(x, lengths)
        loss = criterion(logits, y)

        probs = torch.sigmoid(logits).cpu().numpy()
        probs_all.append(probs)
        labels_all.append(y.cpu().numpy())
        seqs_all.extend(seqs)

        total_loss += loss.item() * x.size(0)
        total_n += x.size(0)

    probs_all = np.concatenate(probs_all, axis=0)
    labels_all = np.concatenate(labels_all, axis=0)
    return total_loss / max(total_n, 1), labels_all, probs_all, seqs_all


def train_one_epoch(model, loader):
    """Run one training epoch for the internal multi-label classifier."""
    model.train()
    total_loss, total_n = 0.0, 0

    for x, lengths, y, _ in loader:
        x = x.to(device)
        lengths = lengths.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x, lengths)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_n += x.size(0)

    return total_loss / max(total_n, 1)

## 5. Train classifier

This section trains the internal multi-label baseline and monitors validation performance.

Expected pattern for this dataset:
- stronger results on common labels such as antibacterial and Gram activity,
- weaker thresholded performance on rare labels such as antiviral, antiparasitic, and anticancer.

The classifier is meant to be useful for screening, not to be treated as a biological ground truth model.


In [ ]:
num_epochs = 30
patience = 7
best_path = MODELS_DIR / "best_external_amp_classifier.pt"

history = []
best_score = -np.inf
wait = 0

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader)
    val_loss, y_val, p_val, _ = predict_loader(model, val_loader)
    val_metrics_df, val_macro = compute_multilabel_metrics(y_val, p_val, threshold=0.5)
    # Use macro AUROC for model selection because it reflects ranking quality
    # across labels without depending on a single fixed threshold.
    monitor = val_macro["macro_auroc"]

    # Reduce learning rate when validation ranking quality plateaus.
    scheduler.step(monitor)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        **val_macro,
    })

    print(
        f"epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"macro_auroc={val_macro['macro_auroc']:.4f} | macro_f1={val_macro['macro_f1']:.4f}"
    )

    if monitor > best_score:
        best_score = monitor
        wait = 0
        # Save the best classifier checkpoint for later reuse in adherence scoring.
        torch.save({
            "model_state_dict": model.state_dict(),
            "target_cols": target_cols,
            "char2idx": char2idx,
            "idx2char": idx2char,
            "max_len": max_len,
            "config": {
                "vocab_size": vocab_size,
                "num_labels": len(target_cols),
            }
        }, best_path)
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping.")
            break

history_df = pd.DataFrame(history)
display(history_df.tail())
print("best checkpoint:", best_path)

In [ ]:
best_path = MODELS_DIR / "best_external_amp_classifier.pt"

In [ ]:
# Load best checkpoint
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])

test_loss, y_test, p_test, seq_test = predict_loader(model, test_loader)
test_metrics_df, test_macro = compute_multilabel_metrics(y_test, p_test, threshold=0.5)

print("Test macro metrics:")
for k, v in test_macro.items():
    print(f"{k:20s} {v:.4f}")

display(test_metrics_df.sort_values("auroc", ascending=False).reset_index(drop=True))

### Baseline result: how good is the classifier?

The baseline is strong enough to be useful as an internal evaluator, but not strong enough to be treated as a biological oracle.

How to read these results:
- ranking metrics such as AUROC indicate whether the classifier separates positives from negatives reasonably well,
- thresholded metrics such as F1 and MCC indicate how reliable binary label decisions are,
- rare labels should be interpreted more cautiously because they are both data-scarce and threshold-sensitive.

For this project, the classifier is mainly used to compare generated conditions relatively, not to make absolute biological claims.


## 6. Threshold tuning per label
A fixed `0.5` threshold is convenient, but it is rarely optimal for imbalanced peptide labels.

Here we tune one threshold per label on the validation split. This matters because downstream condition-adherence depends on binary decisions, not only on ranking quality.


In [ ]:
def tune_thresholds(y_true, y_prob, grid=None):
    """
    Tune one decision threshold per label on the validation set.

    A single threshold of 0.5 is rarely optimal under severe class imbalance,
    so per-label tuning improves the realism of downstream adherence decisions.
    """
    if grid is None:
        grid = np.linspace(0.1, 0.9, 17)

    best_thresholds = {}
    for i, label in enumerate(target_cols):
        yt = y_true[:, i]
        ys = y_prob[:, i]

        best_thr, best_f1 = 0.5, -1
        for thr in grid:
            yp = (ys >= thr).astype(int)
            score = f1_score(yt, yp, zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_thr = float(thr)

        best_thresholds[label] = best_thr
    return best_thresholds

# Tune thresholds only on validation data to avoid leaking test information into model selection.
best_thresholds = tune_thresholds(y_val, p_val)
best_thresholds

In [ ]:
# Recompute test metrics with tuned thresholds
def compute_metrics_with_custom_thresholds(y_true, y_prob, thresholds):
    """
    Recompute per-label and aggregate metrics using label-specific thresholds.
    """
    rows = []
    y_pred = np.zeros_like(y_true, dtype=int)

    for i, label in enumerate(target_cols):
        thr = thresholds[label]
        yt = y_true[:, i]
        ys = y_prob[:, i]
        yp = (ys >= thr).astype(int)
        y_pred[:, i] = yp

        if len(np.unique(yt)) < 2:
            auroc = np.nan
            auprc = np.nan
        else:
            auroc = roc_auc_score(yt, ys)
            auprc = average_precision_score(yt, ys)

        rows.append({
            "label": label,
            "threshold": thr,
            "auroc": auroc,
            "auprc": auprc,
            "f1": f1_score(yt, yp, zero_division=0),
            "precision": precision_score(yt, yp, zero_division=0),
            "recall": recall_score(yt, yp, zero_division=0),
            "mcc": matthews_corrcoef(yt, yp) if len(np.unique(yp)) > 1 else np.nan,
        })

    df_metrics = pd.DataFrame(rows)
    summary = {
        "macro_auroc": np.nanmean(df_metrics["auroc"]),
        "macro_auprc": np.nanmean(df_metrics["auprc"]),
        "macro_f1": np.nanmean(df_metrics["f1"]),
        "macro_precision": np.nanmean(df_metrics["precision"]),
        "macro_recall": np.nanmean(df_metrics["recall"]),
        "macro_mcc": np.nanmean(df_metrics["mcc"]),
        "micro_f1": f1_score(y_true.reshape(-1), y_pred.reshape(-1), zero_division=0),
    }
    return df_metrics, summary, y_pred

test_metrics_tuned_df, test_macro_tuned, y_test_pred_tuned = compute_metrics_with_custom_thresholds(
    y_test, p_test, best_thresholds
)

print("Tuned-threshold test macro metrics:")
for k, v in test_macro_tuned.items():
    print(f"{k:20s} {v:.4f}")

display(test_metrics_tuned_df.sort_values("auroc", ascending=False).reset_index(drop=True))

### Threshold tuning result: why keep the tuned version?

Threshold tuning improves the classifier in the regime that matters for generation evaluation.

**Before vs after tuning**
- `macro F1`: `0.605 -> 0.620`
- `macro recall`: `0.672 -> 0.702`
- `macro MCC`: `0.428 -> 0.432`
- `micro F1`: essentially unchanged (`0.836 -> 0.835`)

**Interpretation**
- The tuned thresholds recover more true positives on rare labels.
- The gain is modest but real, and it comes without changing the model architecture.
- For downstream adherence, tuned thresholds are preferable because the notebook ultimately needs discrete label decisions.

From this point on, the tuned thresholds are the more meaningful screening setup.


## 7. Load generator checkpoint

This section reloads the trained cVAE so the classifier can evaluate generated peptides under controlled conditions.

The classifier and generator remain separate models on purpose:
- the cVAE learns to generate peptide-like sequences,
- the classifier provides an auxiliary signal about target-condition adherence.

This separation is useful because generation loss alone does not tell us whether requested labels are actually expressed in sampled sequences.

In [ ]:
# Accept either best_vae.pt or best_cvae.pt so the notebook stays usable
# across slightly different checkpoint naming conventions.
cvae_ckpt_candidates = [
    MODELS_DIR / "best_vae.pt",
    MODELS_DIR / "best_cvae.pt",
]

cvae_ckpt_path = None
for p in cvae_ckpt_candidates:
    if p.exists():
        cvae_ckpt_path = p
        break

if cvae_ckpt_path is None:
    print("Generator checkpoint not found in models/. Training/evaluation above still works.")
else:
    print("Using generator checkpoint:", cvae_ckpt_path)

In [ ]:
# Rebuild the generator architecture exactly as used in training,
# then load the saved weights for inference-only evaluation.
# cVAE architecture copied from the training notebook
num_layers = 1
dropout = 0.2
latent_dim = 32
enc_hidden_dim = 1024
dec_hidden_dim = 512
embed_dim = 128
cond_dim = 7

condition_cols_vae = [
    "is_antibacterial",
    "is_anti_gram_positive",
    "is_anti_gram_negative",
    "is_antifungal",
    "is_antiviral",
    "is_antiparasitic",
    "is_anticancer",
]

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim, cond_dim, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc_mu = nn.Linear(hidden_dim + cond_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim + cond_dim, latent_dim)

    def forward(self, x, lengths, cond):
        embedded = self.drop(self.embedding(x))
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu().clamp(min=1), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        hidden = h_n[-1]
        hidden_cond = torch.cat([hidden, cond], dim=1)
        return self.fc_mu(hidden_cond), self.fc_logvar(hidden_cond)


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim, cond_dim, dropout=0.2):
        super().__init__()
        self.latent_dim = latent_dim
        self.cond_dim = cond_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            embed_dim + latent_dim + cond_dim, hidden_dim,
            num_layers=num_layers, batch_first=True
        )
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        self.init_h = nn.Linear(latent_dim + cond_dim, hidden_dim)
        self.init_c = nn.Linear(latent_dim + cond_dim, hidden_dim)

    def forward(self, x, z, cond, word_dropout_rate=0.0):
        batch_size, seq_len = x.shape
        z_cond = torch.cat([z, cond], dim=1)
        h0 = torch.tanh(self.init_h(z_cond)).unsqueeze(0)
        c0 = torch.tanh(self.init_c(z_cond)).unsqueeze(0)

        embedded = self.drop(self.embedding(x))
        z_expand = z.unsqueeze(1).expand(-1, seq_len, -1)
        cond_expand = cond.unsqueeze(1).expand(-1, seq_len, -1)
        lstm_input = torch.cat([embedded, z_expand, cond_expand], dim=2)

        outputs, _ = self.lstm(lstm_input, (h0, c0))
        return self.fc_out(outputs)


class CVAE(nn.Module):
    def __init__(self, vocab_size, embed_dim, enc_hidden, dec_hidden, latent_dim, cond_dim, dropout=0.2):
        super().__init__()
        self.encoder = Encoder(vocab_size, embed_dim, enc_hidden, latent_dim, cond_dim, dropout)
        self.decoder = Decoder(vocab_size, embed_dim, dec_hidden, latent_dim, cond_dim, dropout)
        self.latent_dim = latent_dim
        self.cond_dim = cond_dim

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, src, lengths, cond, word_dropout_rate=0.0):
        mu, logvar = self.encoder(src, lengths, cond)
        z = self.reparameterize(mu, logvar)
        logits = self.decoder(src[:, :-1], z, cond, word_dropout_rate=word_dropout_rate)
        target = src[:, 1:]
        return logits, target, mu, logvar

In [ ]:
if cvae_ckpt_path is not None:
    cvae = CVAE(
        vocab_size=vocab_size,
        embed_dim=embed_dim,
        enc_hidden=enc_hidden_dim,
        dec_hidden=dec_hidden_dim,
        latent_dim=latent_dim,
        cond_dim=cond_dim,
        dropout=dropout,
    ).to(device)

    cvae.load_state_dict(torch.load(cvae_ckpt_path, map_location=device, weights_only=True))
    cvae.eval()
    print("Loaded cVAE successfully.")

In [ ]:
def decode_from_z_cond(model, z, cond_t, max_gen_len, temperature=1.0):
    """
    Decode one peptide sequence from a latent sample z under a fixed condition vector.

    Temperature controls how conservative or diverse token sampling is during generation.
    """
    z_cond = torch.cat([z, cond_t], dim=1)
    h = torch.tanh(model.decoder.init_h(z_cond)).unsqueeze(0)
    c = torch.tanh(model.decoder.init_c(z_cond)).unsqueeze(0)

    input_token = torch.LongTensor([[SOS_IDX]]).to(device)
    generated = []

    for _ in range(max_gen_len):
        embedded = model.decoder.embedding(input_token)
        lstm_input = torch.cat([embedded, z.unsqueeze(1), cond_t.unsqueeze(1)], dim=2)
        output, (h, c) = model.decoder.lstm(lstm_input, (h, c))
        logits = model.decoder.fc_out(output.squeeze(1))
        probs = torch.softmax(logits / temperature, dim=-1)
        next_token = torch.multinomial(probs, 1).item()

        if next_token == EOS_IDX:
            break
        if next_token not in [PAD_IDX, SOS_IDX]:
            generated.append(next_token)

        input_token = torch.LongTensor([[next_token]]).to(device)

    seq = "".join(
        idx2char[idx] for idx in generated
        if idx in idx2char and idx2char[idx] not in [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN]
    )
    return seq

def generate_cvae(model, cond, max_gen_len=None, temperature=0.8, alpha=1.0):
    if max_gen_len is None:
        max_gen_len = max_len - 2
    with torch.no_grad():
        cond_t = torch.FloatTensor(cond).unsqueeze(0).to(device) * alpha
        z = torch.randn(1, latent_dim).to(device)
        return decode_from_z_cond(model, z, cond_t, max_gen_len=max_gen_len, temperature=temperature)

## 8. Helper functions for adherence evaluation

The classifier outputs label probabilities for arbitrary sequences.

These helper functions convert raw probabilities into condition-level summaries such as:
- all-requested hit rate,
- exact-match adherence,
- off-target rate,
- requested-label precision and recall.

These metrics are not biological ground truth. They are practical proxies for how well generated peptides follow the requested label pattern.


In [ ]:
@torch.no_grad()
def predict_sequences(model, sequences, batch_size=256):
    """
    Predict multi-label probabilities for an arbitrary list of peptide sequences.
    """
    model.eval()
    probs_out = []

    for start in range(0, len(sequences), batch_size):
        batch = sequences[start:start + batch_size]
        toks = []
        lengths = []
        for seq in batch:
            t, l = tokenize(seq)
            toks.append(t)
            lengths.append(l)
        x = torch.tensor(toks, dtype=torch.long, device=device)
        lengths = torch.tensor(lengths, dtype=torch.long, device=device)

        logits = model(x, lengths)
        probs = torch.sigmoid(logits).cpu().numpy()
        probs_out.append(probs)

    return np.concatenate(probs_out, axis=0) if probs_out else np.zeros((0, len(target_cols)))


def requested_mask_from_condition(cond_name):
    """
    Map a human-readable condition name to the subset of classifier labels
    that should be considered requested for adherence scoring.
    """
    mapping = {
        "antibacterial": "is_antibacterial",
        "gram+": "is_anti_gram_positive",
        "gram-": "is_anti_gram_negative",
        "antifungal": "is_antifungal",
        "anticancer": "is_anticancer",
        "antiviral": "is_antiviral",
        "antiparasitic": "is_antiparasitic",
    }
    mask = np.zeros(len(target_cols), dtype=int)
    for key, col in mapping.items():
        if key in cond_name.lower():
            mask[target_cols.index(col)] = 1
    return mask


def condition_vector(**kwargs):
    order = [
        "is_antibacterial",
        "is_anti_gram_positive",
        "is_anti_gram_negative",
        "is_antifungal",
        "is_antiviral",
        "is_antiparasitic",
        "is_anticancer",
    ]
    return [int(kwargs.get(k, 0)) for k in order]


def adherence_metrics_from_probs(requested_cond_7d, probs, thresholds, return_per_sequence=False):
    # Map requested 7D generator condition to the external classifier label order
    # generator order: antibacterial, gram+, gram-, antifungal, antiviral, antiparasitic, anticancer
    # classifier order: antibacterial, gram+, gram-, antifungal, anticancer, antiviral, antiparasitic
    requested = np.array([
        requested_cond_7d[0],
        requested_cond_7d[1],
        requested_cond_7d[2],
        requested_cond_7d[3],
        requested_cond_7d[6],
        requested_cond_7d[4],
        requested_cond_7d[5],
    ], dtype=int)

    thr_vec = np.array([thresholds[c] for c in target_cols], dtype=float)
    pred_bin = (probs >= thr_vec[None, :]).astype(int)

    requested_idx = np.where(requested == 1)[0]
    off_idx = np.where(requested == 0)[0]

    if len(requested_idx) == 0:
        exact_match = (pred_bin.sum(axis=1) == 0).mean()
        off_target_rate = (pred_bin.sum(axis=1) > 0).mean()
        res = {
            "n_requested_labels": 0,
            "exact_match_rate": float(exact_match),
            "off_target_rate": float(off_target_rate),
            "mean_requested_prob": np.nan,
            "mean_offtarget_prob": float(probs[:, off_idx].mean()) if len(off_idx) else np.nan,
        }
        return (res, pred_bin) if return_per_sequence else res

    req_probs = probs[:, requested_idx]
    req_pred = pred_bin[:, requested_idx]
    off_probs = probs[:, off_idx] if len(off_idx) else np.zeros((len(probs), 0))
    off_pred = pred_bin[:, off_idx] if len(off_idx) else np.zeros((len(probs), 0), dtype=int)

    all_requested_hit = (req_pred.min(axis=1) == 1).mean()
    any_requested_hit = (req_pred.max(axis=1) == 1).mean()
    exact_match = ((req_pred.min(axis=1) == 1) & (off_pred.sum(axis=1) == 0)).mean()
    off_target_rate = (off_pred.sum(axis=1) > 0).mean()

    precision_per_seq = []
    recall_per_seq = []
    f1_per_seq = []

    for i in range(len(pred_bin)):
        y_true = requested
        y_pred = pred_bin[i]
        precision_per_seq.append(precision_score(y_true, y_pred, zero_division=0))
        recall_per_seq.append(recall_score(y_true, y_pred, zero_division=0))
        f1_per_seq.append(f1_score(y_true, y_pred, zero_division=0))

    res = {
        "n_requested_labels": int(len(requested_idx)),
        "all_requested_hit_rate": float(all_requested_hit),
        "any_requested_hit_rate": float(any_requested_hit),
        "exact_match_rate": float(exact_match),
        "off_target_rate": float(off_target_rate),
        "mean_requested_prob": float(req_probs.mean()),
        "mean_offtarget_prob": float(off_probs.mean()) if off_probs.size else np.nan,
        "macro_precision_vs_request": float(np.mean(precision_per_seq)),
        "macro_recall_vs_request": float(np.mean(recall_per_seq)),
        "macro_f1_vs_request": float(np.mean(f1_per_seq)),
    }

    return (res, pred_bin) if return_per_sequence else res

## 9. Generate peptides and score adherence

For each requested condition, we sample peptides from the cVAE and score them with the internal classifier.

This is the main bridge between sequence generation and function-aware evaluation:
it asks not only whether peptides look valid, but whether they express the requested activity pattern according to an independent project-side evaluator.


In [ ]:
# These conditions cover single-label, multi-label, and unconditional generation regimes.
conditions_to_test = {
    "Antibacterial": condition_vector(is_antibacterial=1),
    "Gram+": condition_vector(is_anti_gram_positive=1),
    "Gram-": condition_vector(is_anti_gram_negative=1),
    "Antifungal": condition_vector(is_antifungal=1),
    "Anticancer": condition_vector(is_anticancer=1),
    "Antibacterial + Gram+": condition_vector(is_antibacterial=1, is_anti_gram_positive=1),
    "Antibacterial + Gram-": condition_vector(is_antibacterial=1, is_anti_gram_negative=1),
    "Antifungal + Anticancer": condition_vector(is_antifungal=1, is_anticancer=1),
    "Unconditional": condition_vector(),
}
conditions_to_test

In [ ]:
def generate_batch_for_condition(cvae, cond, n_samples=256, temperature=0.8, alpha=1.0, deduplicate=False):
    """
    Sample a batch of peptides under one requested condition.

    Optional deduplication is useful for candidate review, but for adherence
    estimation it is often informative to keep duplicates and observe sampling behaviour.
    """
    seqs = [generate_cvae(cvae, cond=cond, temperature=temperature, alpha=alpha) for _ in range(n_samples)]
    seqs = [s for s in seqs if isinstance(s, str) and len(s) > 0]
    if deduplicate:
        seqs = list(dict.fromkeys(seqs))
    return seqs

In [ ]:
if cvae_ckpt_path is None:
    print("Skip this section until the generator checkpoint is available.")
else:
    condition_rows = []
    generated_cache = {}

    # Evaluate each condition independently so we can compare how controllable
    # the generator is across simple and composite requests.
    for name, cond in conditions_to_test.items():
        seqs = generate_batch_for_condition(cvae, cond, n_samples=300, temperature=0.8, alpha=1.0, deduplicate=False)
        probs = predict_sequences(model, seqs)
        metrics = adherence_metrics_from_probs(cond, probs, best_thresholds)
        generated_cache[name] = {"cond": cond, "seqs": seqs, "probs": probs, "metrics": metrics}

        row = {"condition": name, "n_generated": len(seqs), **metrics}
        condition_rows.append(row)

    adherence_df = pd.DataFrame(condition_rows).sort_values("condition").reset_index(drop=True)
    display(adherence_df)

### Condition-adherence review

The generator shows useful condition sensitivity on common antibacterial requests, but control is still incomplete and label entanglement remains visible.

How to read the table:
- high `all_requested_hit_rate` means the requested labels are often activated together,
- low `exact_match_rate` means extra non-requested labels are still frequently co-activated,
- `off_target_rate` captures how much unwanted activity remains,
- `macro_f1_vs_request` summarizes overall agreement with the requested label profile.

In practice, this means the generator already responds to conditioning, but it does not yet produce cleanly disentangled activity profiles.


## 10. Inspect top candidates
The ranking step is useful for practical screening.

In general, the best candidates are peptides that:
- satisfy all requested labels
- keep off-target probability low
- keep requested-label confidence high
- remain interesting after deduplication and novelty checks


In [ ]:
def rank_generated_candidates(seqs, probs, requested_cond_7d, thresholds, top_k=20):
    """
    Rank generated peptides for practical triage.

    The ranking favours sequences with high confidence on requested labels
    and low confidence on off-target labels, while also exposing exact-match
    and hit-style indicators for manual review.
    """
    requested = np.array([
        requested_cond_7d[0],
        requested_cond_7d[1],
        requested_cond_7d[2],
        requested_cond_7d[3],
        requested_cond_7d[6],
        requested_cond_7d[4],
        requested_cond_7d[5],
    ], dtype=int)

    req_idx = np.where(requested == 1)[0]
    off_idx = np.where(requested == 0)[0]

    rows = []
    thr_vec = np.array([thresholds[c] for c in target_cols], dtype=float)
    pred_bin = (probs >= thr_vec[None, :]).astype(int)

    for seq, p, pred in zip(seqs, probs, pred_bin):
        req_score = float(p[req_idx].mean()) if len(req_idx) else 0.0
        off_score = float(p[off_idx].mean()) if len(off_idx) else 0.0
        rows.append({
            "sequence": seq,
            "length": len(seq),
            "requested_mean_prob": req_score,
            "offtarget_mean_prob": off_score,
            "exact_match": int(np.array_equal(pred, requested)),
            **{f"prob_{lab}": float(val) for lab, val in zip(target_cols, p)},
        })

    ranked = pd.DataFrame(rows).sort_values(
        ["exact_match", "requested_mean_prob", "offtarget_mean_prob"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    return ranked.head(top_k)

In [ ]:
if cvae_ckpt_path is not None:
    # Inspect one representative condition in detail to see which generated peptides
    # look most promising under the internal screening criterion.
    example_name = "Antibacterial + Gram+"
    ranked = rank_generated_candidates(
        generated_cache[example_name]["seqs"],
        generated_cache[example_name]["probs"],
        generated_cache[example_name]["cond"],
        best_thresholds,
        top_k=15,
    )
    display(ranked)

## 11. Save artifacts

This section saves the classifier checkpoint, tuned thresholds, and condition-adherence summaries so the same evaluator can be reused later without retraining.

These artifacts are also useful for the final report, where stable exported tables are preferable to notebook-only outputs.


In [ ]:
artifacts = {
    "target_cols": target_cols,
    "best_thresholds": best_thresholds,
    "test_macro_default": test_macro,
    "test_macro_tuned": test_macro_tuned,
}

with open(MODELS_DIR / "external_amp_classifier_artifacts.json", "w") as f:
    # Save the minimum metadata required to reuse the classifier consistently
    # in later notebooks and report-generation scripts.
    json.dump(artifacts, f, indent=2)

if "adherence_df" in globals():
    adherence_df.to_csv(MODELS_DIR / "condition_adherence_metrics.csv", index=False)

print("Saved:")
print(" -", best_path)
print(" -", MODELS_DIR / "external_amp_classifier_artifacts.json")
if "adherence_df" in globals():
    print(" -", MODELS_DIR / "condition_adherence_metrics.csv")

## 12. Performance plots

The final figures summarize two things:
1. whether classifier training was stable,
2. whether the classifier is usable as a downstream evaluator for generated peptides.


In [ ]:
sns.set_theme(style="whitegrid", context="talk")

# Combine optimization curves, per-label test quality, and adherence summaries
# into one compact visual overview for the final report.
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

axes[0, 0].plot(history_df["epoch"], history_df["train_loss"], marker="o", linewidth=2, label="Train loss")
axes[0, 0].plot(history_df["epoch"], history_df["val_loss"], marker="o", linewidth=2, label="Val loss")
axes[0, 0].set_title("Loss curves")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend()

metric_cols = [c for c in ["macro_auroc", "macro_f1", "micro_f1"] if c in history_df.columns]
for col in metric_cols:
    axes[0, 1].plot(history_df["epoch"], history_df[col], marker="o", linewidth=2, label=col)
axes[0, 1].set_title("Validation metrics")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Score")
axes[0, 1].set_ylim(0.0, 1.0)
axes[0, 1].legend()

auroc_plot_df = test_metrics_df[["label", "auroc"]].sort_values("auroc", ascending=False)
sns.barplot(data=auroc_plot_df, x="auroc", y="label", color="#4C72B0", ax=axes[1, 0])
axes[1, 0].set_title("Per-label AUROC on test set")
axes[1, 0].set_xlabel("AUROC")
axes[1, 0].set_ylabel("")
axes[1, 0].set_xlim(0.0, 1.0)

f1_compare_df = pd.concat([
    test_metrics_df[["label", "f1"]].assign(setting="default threshold"),
    test_metrics_tuned_df[["label", "f1"]].assign(setting="tuned threshold"),
], ignore_index=True)
label_order = (
    f1_compare_df[f1_compare_df["setting"] == "tuned threshold"]
    .sort_values("f1", ascending=False)["label"]
)
sns.barplot(
    data=f1_compare_df,
    x="f1",
    y="label",
    hue="setting",
    order=label_order,
    palette="Set2",
    ax=axes[1, 1],
)
axes[1, 1].set_title("Per-label F1 before vs after threshold tuning")
axes[1, 1].set_xlabel("F1")
axes[1, 1].set_ylabel("")
axes[1, 1].set_xlim(0.0, 1.0)
axes[1, 1].legend(title="")

plt.tight_layout()
plt.show()

### Plot review and baseline interpretation

The figure should be read as a compact summary of both classifier quality and downstream screening usefulness.

Training curves:
- smooth train/validation behaviour suggests the classifier is stable enough for internal screening,
- the goal is not perfect accuracy, but a dependable auxiliary signal.

Per-label AUROC:
- common labels should generally rank better than rare ones,
- weak rare-label performance should be acknowledged explicitly in the report.

Adherence panel:
- strong requested hit rates indicate condition sensitivity,
- weaker exact-match behaviour indicates remaining label entanglement and off-target activation.


## 13. Future work
This baseline is already useful for screening, but it should not be treated as a final biological validator.

High-value next steps:
1. Replace the random split with a **homology-aware split**.
2. Calibrate probabilities with temperature scaling, isotonic regression, or Platt scaling.
3. Add external toxicity and hemolysis predictors to turn adherence into a safer multi-objective screen.
4. Compare against stronger sequence encoders such as CNN or protein-language-model embeddings.
5. Use the classifier inside reranking or latent-space optimization loops instead of only post-hoc evaluation.
